#🎯 Medidas de variabilidade

Até agora vimos:
- média → onde os dados estão
- variabilidade → o quanto eles se espalham

Agora vamos responder outra pergunta:

👉 "Onde um valor específico está dentro da distribuição?"

É isso que as medidas de separação fazem.

#🧸  Exemplo conceitual inicial

As medidas de separação dividem os dados ordenados em partes iguais.

- Percentis → 100 partes
- Quartis → 4 partes

In [ ]:
import pandas as pd

dados = pd.Series([1, 2, 3, 4, 5, 6, 7, 8, 9])

print("P50 (mediana):", dados.quantile(0.5))
print("P25 (Q1):", dados.quantile(0.25))
print("P75 (Q3):", dados.quantile(0.75))

P50 (mediana): 5.0
P25 (Q1): 3.0
P75 (Q3): 7.0


- P50 → metade dos dados está abaixo
- P25 → 25% dos dados está abaixo
- P75 → 75% dos dados está abaixo

Ou seja:
essas medidas mostram "posição", não média

#❤️  Aplicação em um dataset de exemplo em saúde

Neste exemplo simulado, observamos que uma pequena parcela dos beneficiários concentra a maior parte do custo.

Esse padrão é comum em saúde e reforça a importância de identificar e acompanhar os pacientes de maior risco e maior impacto financeiro.

In [ ]:
import pandas as pd
import numpy as np

def gerar_dataset_sinistro_beneficiarios(
    n_benef=1000,
    seed=42
):
    np.random.seed(seed)

    # =========================
    # 1) IDs
    # =========================
    ids = [f"BENE_{i:05d}" for i in range(1, n_benef + 1)]

    # =========================
    # 2) Distribuição de custos (mistura)
    # =========================
    # grupo 1: maioria - baixo custo
    n_baixo = int(n_benef * 0.85)
    baixo = np.random.gamma(shape=2, scale=300, size=n_baixo)

    # grupo 2: médio custo
    n_medio = int(n_benef * 0.10)
    medio = np.random.gamma(shape=2, scale=1500, size=n_medio)

    # grupo 3: alto custo (outliers relevantes)
    n_alto = n_benef - n_baixo - n_medio
    alto = np.random.gamma(shape=2, scale=8000, size=n_alto)

    custos = np.concatenate([baixo, medio, alto])

    # embaralhar
    np.random.shuffle(custos)

    # =========================
    # 3) número de eventos (opcional)
    # =========================
    eventos = np.random.poisson(lam=3, size=n_benef) + 1

    # =========================
    # 4) dataset final
    # =========================
    df = pd.DataFrame({
        "Id Beneficiário": ids,
        "Qtd_Eventos": eventos,
        "Soma_Sinistro_Total": custos
    })

    return df

In [ ]:
df_pareto_usuario = gerar_dataset_sinistro_beneficiarios(n_benef=1000)

df_pareto_usuario.head()

,Id Beneficiário,Qtd_Eventos,Soma_Sinistro_Total
0,BENE_00001,3,1014.587929
1,BENE_00002,6,18.131648
2,BENE_00003,2,592.064998
3,BENE_00004,1,431.025898
4,BENE_00005,3,635.692359


In [ ]:
df_pareto_usuario.shape

(1000, 3)

In [ ]:
df_pareto_usuario['Id Beneficiário'].nunique()

1000

# 📊 Cálculo das medidas de separação

In [ ]:
df_pareto_usuario["Soma_Sinistro_Total"].describe()

,Soma_Sinistro_Total
count,1000.000000
mean,1727.717646
std,4579.272025
min,18.131648
25%,328.015060
50%,593.714905
75%,1070.669580
max,44154.616392


In [ ]:
custos = df_pareto_usuario["Soma_Sinistro_Total"]

percentis = pd.DataFrame({
    "Percentil": ["P10", "P25", "P50", "P75", "P90", "P95"],
    "Valor": [
        custos.quantile(0.10),
        custos.quantile(0.25),
        custos.quantile(0.50),
        custos.quantile(0.75),
        custos.quantile(0.90),
        custos.quantile(0.95),
    ]
})

percentis

,Percentil,Valor
0,P10,194.650175
1,P25,328.015060
2,P50,593.714905
3,P75,1070.669580
4,P90,2622.417712
5,P95,6367.870199


Exemplo de leitura:

- P50 (mediana) → custo típico
- P75 → 25% dos beneficiários estão acima desse valor
- P90 → apenas 10% estão acima
- P95 → elite de alto custo

👉 quanto maior o percentil, mais “extremo” é o grupo

In [ ]:
df_pareto_usuario.columns

Index(['Id Beneficiário', 'Qtd_Eventos', 'Soma_Sinistro_Total'], dtype='object')

In [ ]:
import plotly.express as px

fig = px.histogram(
    df_pareto_usuario,
    x="Soma_Sinistro_Total",
    nbins=50,
    marginal="box",
    title="Distribuição com percentis"
)

# adicionar linhas
for p, label in zip(
    [0.25, 0.5, 0.75],
    ["Q1", "Mediana", "Q3"]
):
    fig.add_vline(
        x=custos.quantile(p),
        line_dash="dash"
    )

fig.update_layout(template="plotly_white")

fig.show()

- a caixa (boxplot) mostra os quartis
- a linha central → mediana
- limites da caixa → Q1 e Q3
- caudas → valores extremos

👉 você está literalmente vendo as separações dos dados

# 🧠 Dividindo os beneficiários por percentil

O qcut divide os dados em grupos com o mesmo número de observações. Ou seja: cada grupo representa uma faixa de percentil


* 4 grupos → quartis
* 5 grupos → quintis
* 10 grupos → decis
* 100 grupos → percentis

##### **Quartis**

In [ ]:
df_pareto_usuario["quartil"] = pd.qcut(
    df_pareto_usuario["Soma_Sinistro_Total"],
    q=4,
    labels=["Q1 (baixo)", "Q2", "Q3", "Q4 (alto)"]
)

df_pareto_usuario["quartil"].value_counts()

,count
quartil,
Q1 (baixo),250
Q2,250
Q3,250
Q4 (alto),250


- Q1 → 25% com menor custo
- Q4 → 25% com maior custo

👉 já é uma separação direta por posição

In [ ]:
resumo_quartil = (
    df_pareto_usuario
    .groupby("quartil")["Soma_Sinistro_Total"]
    .agg(["count", "sum"])
    .sort_index()
)

resumo_quartil["pct_custo"] = resumo_quartil["sum"] / resumo_quartil["sum"].sum()

/tmp/ipykernel_267/2657410872.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
resumo_quartil_fmt = resumo_quartil.copy()

resumo_quartil_fmt["sum"] = resumo_quartil_fmt["sum"].apply(
    lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

resumo_quartil_fmt["pct_custo"] = resumo_quartil_fmt["pct_custo"].apply(
    lambda x: f"{x:.1%}"
)

resumo_quartil_fmt

,count,sum,pct_custo
quartil,,,
Q1 (baixo),250,R$ 52.079,3.0%
Q2,250,R$ 115.338,6.7%
Q3,250,R$ 198.282,11.5%
Q4 (alto),250,R$ 1.362.018,78.8%


##### **Decis**

In [ ]:
df_pareto_usuario["decil"] = pd.qcut(
    df_pareto_usuario["Soma_Sinistro_Total"],
    q=10,
    labels=[f"D{i}" for i in range(1, 11)]
)

df_pareto_usuario["decil"].value_counts()

,count
decil,
D1,100
D2,100
D3,100
D4,100
D5,100
D6,100
D7,100
D8,100
D9,100


In [ ]:
resumo_decil = (
    df_pareto_usuario
    .groupby("decil")["Soma_Sinistro_Total"]
    .agg(["count", "sum"])
    .sort_index()
)

resumo_decil["pct_custo"] = resumo_decil["sum"] / resumo_decil["sum"].sum()

/tmp/ipykernel_267/448197609.py:3: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [ ]:
resumo_decil_fmt = resumo_decil.copy()

resumo_decil_fmt["sum"] = resumo_decil_fmt["sum"].apply(
    lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

resumo_decil_fmt["pct_custo"] = resumo_decil_fmt["pct_custo"].apply(
    lambda x: f"{x:.1%}"
)

resumo_decil_fmt

,count,sum,pct_custo
decil,,,
D1,100,R$ 12.014,0.7%
D2,100,R$ 24.548,1.4%
D3,100,R$ 33.335,1.9%
D4,100,R$ 43.644,2.5%
D5,100,R$ 53.877,3.1%
D6,100,R$ 66.074,3.8%
D7,100,R$ 82.120,4.8%
D8,100,R$ 109.172,6.3%
D9,100,R$ 174.548,10.1%


# 🧠 Classificando beneficiários por percentil

In [ ]:
df_pareto_usuario["percentil"] = df_pareto_usuario["Soma_Sinistro_Total"].rank(pct=True)

In [ ]:
df_pareto_usuario.head(3)

,Id Beneficiário,Qtd_Eventos,Soma_Sinistro_Total,quartil,decil,percentil
0,BENE_00001,3,1014.587929,Q3,D8,0.729
1,BENE_00002,6,18.131648,Q1 (baixo),D1,0.001
2,BENE_00003,2,592.064998,Q2,D5,0.500


In [ ]:
def classificar(p):
    if p <= 0.5:
        return "Baixo custo"
    elif p <= 0.8:
        return "Médio custo"
    elif p <= 0.95:
        return "Alto custo"
    else:
        return "Altíssimo custo"

df_pareto_usuario["faixa_custo"] = df_pareto_usuario["percentil"].apply(classificar)

Agora cada beneficiário tem uma posição relativa na distribuição.

Isso permite:
- segmentar população
- priorizar cuidado
- identificar pacientes críticos

In [ ]:
df_pareto_usuario["faixa_custo"].value_counts()

,count
faixa_custo,
Baixo custo,500
Médio custo,300
Alto custo,150
Altíssimo custo,50


In [ ]:
df_pareto_usuario["faixa_custo"].value_counts(normalize=True) * 100

,proportion
faixa_custo,
Baixo custo,50.0
Médio custo,30.0
Alto custo,15.0
Altíssimo custo,5.0


# 📈 Visualizando curvas de concentração

Nos percentis tradicionais, perguntamos:

"qual é o valor abaixo do qual está uma certa fração dos dados?"

Na curva de concentração, perguntamos:

"qual é a fração dos beneficiários que acumula uma certa fração dos dados?"

Ambas as abordagens usam a lógica de posição acumulada dentro de uma distribuição.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

def curva_concentracao_sinistro_beneficiario(
    df_benef: pd.DataFrame,
    col_id: str = "Id Beneficiário",
    col_valor: str = "Soma_Sinistro_Total",
    titulo: str = "Curva de concentração do sinistro por beneficiário",
    alvo: float = 0.80,
):
    # 1) preparação
    d = df_benef[[col_id, col_valor]].copy()

    d[col_valor] = pd.to_numeric(d[col_valor], errors="coerce").fillna(0.0)
    d = d.dropna(subset=[col_id]).copy()

    # garante unicidade por beneficiário
    d = d.groupby(col_id, as_index=False)[col_valor].sum()

    # ordena do maior para o menor
    d = d.sort_values(col_valor, ascending=False).reset_index(drop=True)

    n = len(d)
    if n == 0:
        raise ValueError("DataFrame vazio após filtros.")

    total = float(d[col_valor].sum())
    if total <= 0:
        raise ValueError("Soma total do sinistro é zero.")

    # 2) curvas acumuladas
    d["pct_beneficiarios"] = np.arange(1, n + 1) / n
    d["pct_sinistro"] = d[col_valor].cumsum() / total

    # 3) ponto-alvo
    idx = (d["pct_sinistro"] >= alvo).idxmax()

    pct_benef = d.loc[idx, "pct_beneficiarios"]
    pct_sin = d.loc[idx, "pct_sinistro"]
    qtd_beneficiarios = idx + 1
    valor_abs = d.loc[:idx, col_valor].sum()

    def fmt_ptbr(v, casas=2):
        s = f"{v:,.{casas}f}"
        return s.replace(",", "X").replace(".", ",").replace("X", ".")

    texto_anotacao = (
        f"<b>{pct_benef:.1%} dos beneficiários</b><br>"
        f"concentram <b>{pct_sin:.1%}</b> do sinistro<br>"
        f"({qtd_beneficiarios} beneficiários)<br>"
        f"R$ {fmt_ptbr(valor_abs, 2)}"
    )

    # 4) gráfico
    fig = px.line(
        d,
        x="pct_beneficiarios",
        y="pct_sinistro",
        labels={
            "pct_beneficiarios": "% acumulado de beneficiários",
            "pct_sinistro": "% acumulado do sinistro",
        },
        title=titulo
    )

    # linha de igualdade
    fig.add_scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash", width=2, color="gray"),
        name="Igualdade"
    )

    # linha horizontal no alvo
    fig.add_hline(
        y=alvo,
        line_dash="dot",
        line_width=2,
        annotation_text=f"Alvo: {alvo:.0%} do sinistro",
        annotation_position="top left"
    )

    # linha vertical no percentual de beneficiários
    fig.add_vline(
        x=pct_benef,
        line_dash="dot",
        line_width=2,
        #annotation_text=f"{pct_benef:.0%} dos beneficiários",
        #annotation_position="top right"
    )

    # ponto destacado
    fig.add_scatter(
        x=[pct_benef],
        y=[pct_sin],
        mode="markers",
        marker=dict(size=10, color="red"),
        name="Ponto de concentração"
    )

    # anotação
    fig.add_annotation(
        x=pct_benef,
        y=pct_sin,
        text=texto_anotacao,
        showarrow=True,
        arrowhead=2,
        ax=80,
        ay=-60,
        bgcolor="white"
    )

    # layout
    fig.update_layout(
        xaxis=dict(tickformat=".0%", range=[0, 1]),
        yaxis=dict(tickformat=".0%", range=[0, 1]),
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend_title_text="",
        width=1300,
        height=700,
    )

    fig.show()

    return fig, d, {
        "percentual_beneficiarios": pct_benef,
        "quantidade_beneficiarios": qtd_beneficiarios,
        "percentual_sinistro": pct_sin,
        "valor_absoluto": valor_abs
    }

In [ ]:
fig, df_conc, ponto = curva_concentracao_sinistro_beneficiario(
    df_pareto_usuario,
    col_id="Id Beneficiário",
    col_valor="Soma_Sinistro_Total",
    titulo="Curva de concentração do sinistro por beneficiário",
    alvo=0.80
)

ponto

{'percentual_beneficiarios': np.float64(0.27),
 'quantidade_beneficiarios': 270,
 'percentual_sinistro': np.float64(0.8003823175791792),
 'valor_absoluto': np.float64(1382834.653728686)}

A curva mostra que o sinistro não está distribuído de forma homogênea entre os beneficiários.

O ponto destacado indica qual percentual acumulado de beneficiários é necessário para atingir 80% do sinistro total.

Quanto menor esse percentual, maior a concentração do custo em poucos beneficiários.

Essa informação é útil para:
- segmentação de risco
- priorização de cuidado
- desenho de programas de gestão de casos
- monitoramento de alto custo

In [ ]:
import plotly.graph_objects as go

fig = px.line(
    df_conc,
    x="pct_beneficiarios",
    y="pct_sinistro",
    title="Curva de concentração com leitura por percentis"
)

# linha de igualdade
fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    line=dict(dash="dash", color="gray"),
    name="Igualdade"
)

# marcar percentis importantes
for p in [0.5, 0.8, 0.9]:
    y_val = df_conc.loc[
        (df_conc["pct_beneficiarios"] - p).abs().idxmin(),
        "pct_sinistro"
    ]

    fig.add_scatter(
        x=[p],
        y=[y_val],
        mode="markers+text",
        text=[f"P{int(p*100)}"],
        textposition="top center"
    )

fig.update_layout(
    xaxis=dict(tickformat=".0%"),
    yaxis=dict(tickformat=".0%"),
    template="plotly_white",
    width=1300,
    height=700,
)

fig.show()

Percentil responde:
"onde está um indivíduo na distribuição?"

Curva de concentração responde:
"quanto impacto esse grupo tem no total?"

👉 juntas, elas mostram:

posição + impacto

# 🔥 Takeaway

Se tiver que levar uma coisa dessa aula:

- percentis mostram posição relativa
- quartis dividem os dados em 4 partes
- ajudam a identificar concentração dos dados
- são essenciais para análise de distribuição
- permitem decisões mais segmentadas
- média responde "quanto"
- percentil responde "onde"
